<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/01_extraccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Extracción de datos crudos

Extrae texto plano (`text`) y etiqueta (`label`) desde las 3 fuentes de
entrenamiento en su formato original:

- **phishing_pot** (.eml) — github.com/rf-peixoto/phishing_pot
- **Nazario** (.mbox) — monkey.org/~jose/phishing
- **Enron** (carpetas de correo) — cs.cmu.edu/~enron

Cada fuente se procesa por separado y se guarda como CSV intermedio,
antes de pasar al notebook `02_union.ipynb`.

## 1. Descarga de phishing_pot (GitHub)

In [1]:
# Clonamos el repositorio completo porque GitHub no permite descargar
# una sola carpeta sin clonar todo el repo.
!git clone https://github.com/rf-peixoto/phishing_pot.git

Cloning into 'phishing_pot'...
remote: Enumerating objects: 10095, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 10095 (delta 230), reused 207 (delta 207), pack-reused 9834 (from 2)
Receiving objects: 100% (10095/10095), 203.71 MiB | 18.50 MiB/s, done.
Resolving deltas: 100% (2498/2498), done.
Updating files: 100% (8619/8619), done.


## 2. Verificar estructura de carpetas de phishing_pot

Antes de escribir el código de extracción, necesitamos confirmar en qué
subcarpeta están los archivos `.eml` reales (no asumirlo a ciegas).

In [3]:
import os

# Recorremos el repo clonado mostrando solo los primeros 2 niveles
# de profundidad, para no saturar la salida con miles de archivos.
for root, dirs, files in os.walk('phishing_pot'):
    nivel = root.replace('phishing_pot', '').count(os.sep)
    indent = ' ' * 2 * nivel
    print(f'{indent}{os.path.basename(root)}/')
    if nivel < 2:
        for f in files[:5]:  # solo mostramos 5 archivos de ejemplo por carpeta
            print(f'{indent}  {f}')

phishing_pot/
  LICENSE
  README.md
  img/
    phishing_pot.png
  email/
    sample-2379.eml
    sample-777.eml
    sample-6945.eml
    sample-7898.eml
    sample-1502.eml
  .github/
    FUNDING.yml
  .git/
    HEAD
    description
    config
    index
    packed-refs
    info/
    logs/
      refs/
        heads/
        remotes/
          origin/
    hooks/
    refs/
      tags/
      heads/
      remotes/
        origin/
    branches/
    objects/
      info/
      pack/


## 3. Extraer texto de los archivos .eml de phishing_pot

Cada archivo .eml contiene un correo completo (headers + cuerpo).
Usamos la librería estándar `email` de Python para parsear cada archivo
y extraer el asunto (subject) y el cuerpo (body) por separado, siguiendo
el mismo esquema `text = subject + " " + body` que ya usaste para las
otras fuentes.

Etiqueta: todos los correos de este repositorio son phishing real
recolectado vía honeypots (ver README del repo) → `label = 1`.

In [4]:
import os
import email
from email import policy
import pandas as pd

RUTA_EML = 'phishing_pot/email'

def extraer_texto_eml(ruta_archivo):
    """
    Abre un archivo .eml, extrae el asunto y el cuerpo de texto plano,
    y los combina en un solo campo 'text'. Si el correo tiene varias
    partes (multipart), recorre todas las partes de texto plano y las
    concatena, ignorando adjuntos binarios (imágenes, PDFs, etc.).
    """
    with open(ruta_archivo, 'rb') as f:
        msg = email.message_from_binary_file(f, policy=policy.default)

    subject = msg.get('Subject', '')

    cuerpo = ''
    if msg.is_multipart():
        for parte in msg.walk():
            if parte.get_content_type() == 'text/plain':
                try:
                    cuerpo += parte.get_content()
                except Exception:
                    pass  # si una parte falla al decodificar, se ignora y se sigue
    else:
        try:
            cuerpo = msg.get_content()
        except Exception:
            cuerpo = ''

    return f"{subject} {cuerpo}".strip()


# Recorremos todos los archivos .eml de la carpeta
filas = []
errores = 0
for nombre_archivo in os.listdir(RUTA_EML):
    if nombre_archivo.endswith('.eml'):
        ruta_completa = os.path.join(RUTA_EML, nombre_archivo)
        try:
            texto = extraer_texto_eml(ruta_completa)
            filas.append({'text': texto, 'label': 1})
        except Exception as e:
            errores += 1
            print(f'Error en {nombre_archivo}: {e}')

df_pot = pd.DataFrame(filas)
print(f'Archivos procesados correctamente: {len(df_pot)}')
print(f'Archivos con error: {errores}')
print()
print(df_pot.head(3))

Archivos procesados correctamente: 8614
Archivos con error: 0

                                                text  label
0  Hiba from Ledger [Ledger](http://links.iterabl...      1
1  [Action Needed] Revise Payment Information - J...      1
2  Herzlichen Glückwunsch! Ihr Edeka-Sofortgewinn...      1


## Extracción de Nazario Phishing Corpus (.mbox)

El corpus está distribuido en archivos .mbox separados por año (2015-2025).
A diferencia de phishing_pot (un archivo .eml por correo), cada archivo
.mbox contiene MUCHOS correos concatenados en un solo archivo de texto.

Se descargan los años 2015-2025, siguiendo el estándar más común en la
literatura (varios papers descartan años anteriores a 2015 porque el
phishing antiguo no representa bien las tácticas actuales).

In [5]:
# Descargamos los archivos .mbox por año desde el sitio oficial de Jose Nazario.
import urllib.request
import os

BASE_URL = 'https://monkey.org/~jose/phishing/'
ANIOS = list(range(2015, 2026))  # 2015 a 2025 inclusive

os.makedirs('nazario_mbox', exist_ok=True)

for anio in ANIOS:
    nombre_archivo = f'phishing-{anio}'
    url = BASE_URL + nombre_archivo
    destino = f'nazario_mbox/{nombre_archivo}'
    try:
        urllib.request.urlretrieve(url, destino)
        tamanio_mb = os.path.getsize(destino) / (1024 * 1024)
        print(f'Descargado {nombre_archivo} ({tamanio_mb:.1f} MB)')
    except Exception as e:
        print(f'Error descargando {nombre_archivo}: {e}')

Descargado phishing-2015 (8.4 MB)
Descargado phishing-2016 (12.0 MB)
Descargado phishing-2017 (10.8 MB)
Descargado phishing-2018 (8.4 MB)
Descargado phishing-2019 (5.0 MB)
Descargado phishing-2020 (5.6 MB)
Descargado phishing-2021 (2.6 MB)
Descargado phishing-2022 (4.6 MB)
Descargado phishing-2023 (7.4 MB)
Descargado phishing-2024 (14.7 MB)
Descargado phishing-2025 (17.9 MB)


## Extraer texto de los archivos .mbox de Nazario

A diferencia de los .eml (un correo por archivo), un .mbox contiene
muchos correos concatenados. Usamos el módulo `mailbox` de Python
(el mismo que reporta el paper "Phishing Codebook" para este corpus),
que sabe separar automáticamente cada correo individual dentro del
archivo .mbox.

Etiqueta: todo el corpus de Nazario es phishing real recolectado del
inbox del autor → `label = 1`.

Siguiendo la buena práctica reportada en la literatura, asignamos un
ID único por correo (año + índice), para mantener trazabilidad.

In [8]:
import mailbox
import pandas as pd

filas = []
errores = 0

for anio in ANIOS:
    ruta_mbox = f'nazario_mbox/phishing-{anio}'
    try:
        buzon = mailbox.mbox(ruta_mbox)
    except Exception as e:
        print(f'No se pudo abrir phishing-{anio}: {e}')
        continue

    contador_anio = 0
    for msg in buzon:
        try:
            subject = msg.get('Subject', '')

            cuerpo = ''
            if msg.is_multipart():
                for parte in msg.walk():
                    if parte.get_content_type() == 'text/plain':
                        try:
                            payload = parte.get_payload(decode=True)
                            if payload:
                                cuerpo += payload.decode('utf-8', errors='ignore')
                        except Exception:
                            pass
            else:
                payload = msg.get_payload(decode=True)
                if payload:
                    cuerpo = payload.decode('utf-8', errors='ignore')

            texto = f"{subject} {cuerpo}".strip()

            filas.append({'text': texto, 'label': 1})
            contador_anio += 1
        except Exception as e:
            errores += 1

    print(f'phishing-{anio}: {contador_anio} correos extraídos')

df_nazario = pd.DataFrame(filas)
print()
print(f'Total correos extraídos: {len(df_nazario)}')
print(f'Errores: {errores}')
print()
print(df_nazario.head(5))

phishing-2015: 307 correos extraídos
phishing-2016: 494 correos extraídos
phishing-2017: 325 correos extraídos
phishing-2018: 288 correos extraídos
phishing-2019: 243 correos extraídos
phishing-2020: 158 correos extraídos
phishing-2021: 101 correos extraídos
phishing-2022: 247 correos extraídos
phishing-2023: 419 correos extraídos
phishing-2024: 403 correos extraídos
phishing-2025: 481 correos extraídos

Total correos extraídos: 3466
Errores: 0

                                                text  label
0  DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL D...      1
1  Important Security Update <HTML><BODY><P>\n<TA...      1
2  Important Security Update <HTML><BODY><P>\n<TA...      1
3  Important Security Update \n   \t \n   Dear Cu...      1
4  Important Security Update \n   \t \n   Dear Cu...      1


## Extracción de Enron Email Dataset (carpetas de correo)

El dataset de Enron viene como un archivo .tar.gz (~1.7 GB) con ~150
buzones de usuarios distintos, organizados en carpetas. Cada correo
es un archivo de texto plano individual (no .eml con headers MIME
completos, formato más simple).

Dado el volumen (~500,000 correos), se aplica un submuestreo aleatorio
(random subsampling; Susan & Kumar, 2021) de aproximadamente 5,000
correos, para no desbalancear el dataset frente a las ~12,000 filas de
phishing ya extraídas de las otras 2 fuentes.

Etiqueta: todo el corpus de Enron es correspondencia corporativa
legítima → `label = 0`.

In [9]:
# Descargamos el dataset completo de Enron desde la fuente oficial (CMU).
import urllib.request
import os

URL_ENRON = 'https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz'
DESTINO = 'enron_mail.tar.gz'

if not os.path.exists(DESTINO):
    print('Descargando Enron (~1.7 GB, puede tardar varios minutos)...')
    urllib.request.urlretrieve(URL_ENRON, DESTINO)
    print('Descarga completa.')
else:
    print('El archivo ya existe, se omite la descarga.')

print('Tamaño:', round(os.path.getsize(DESTINO) / (1024**3), 2), 'GB')

Descargando Enron (~1.7 GB, puede tardar varios minutos)...
Descarga completa.
Tamaño: 0.41 GB


In [12]:
# Descomprimimos en almacenamiento temporal de Colab.
import tarfile

print('Descomprimiendo...')
with tarfile.open('enron_mail.tar.gz', 'r:gz') as tar:
    tar.extractall('enron_extracted')
print('Descompresión completa.')

Descomprimiendo...


/tmp/ipykernel_1094/433288161.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('enron_extracted')


Descompresión completa.


## Verificar estructura de carpetas extraídas

Antes de hacer el muestreo, confirmamos que la descompresión generó la
estructura esperada: una carpeta por usuario, cada una con subcarpetas
de correo.

In [13]:
import os

ruta_maildir = 'enron_extracted/maildir'
usuarios = os.listdir(ruta_maildir)
print('Cantidad de usuarios/buzones:', len(usuarios))
print('Primeros 5 usuarios:', usuarios[:5])

Cantidad de usuarios/buzones: 150
Primeros 5 usuarios: ['pimenov-v', 'haedicke-m', 'reitmeyer-j', 'donoho-l', 'crandell-s']


## Muestreo aleatorio y extracción de texto de Enron

Recorremos todas las carpetas de usuarios, listamos todos los archivos
de correo disponibles, y tomamos una muestra aleatoria de ~5,000 antes
de extraer el texto — así no perdemos tiempo procesando los ~500,000
correos completos cuando solo necesitamos una fracción balanceada.

Cada archivo de correo de Enron es texto plano simple con headers
("Subject:", etc.) seguidos del cuerpo, sin codificación MIME compleja
como en .eml. Se extrae el asunto y el cuerpo de la misma forma que en
las otras 2 fuentes: `text = subject + " " + body`.

Etiqueta: `label = 0` (correspondencia legítima).

In [17]:
import os
import random

random.seed(42)  # reproducibilidad: misma muestra si se vuelve a correr

ruta_maildir = 'enron_extracted/maildir'

# Recolectamos las rutas de TODOS los archivos de correo disponibles
todas_las_rutas = []
for usuario in os.listdir(ruta_maildir):
    ruta_usuario = os.path.join(ruta_maildir, usuario)
    for carpeta_raiz, _, archivos in os.walk(ruta_usuario):
        for archivo in archivos:
            todas_las_rutas.append(os.path.join(carpeta_raiz, archivo))

print('Total de correos disponibles en Enron:', len(todas_las_rutas))

# Tomamos una muestra aleatoria de 12080
N_MUESTRA = 12080
muestra_rutas = random.sample(todas_las_rutas, min(N_MUESTRA, len(todas_las_rutas)))
print('Tamaño de la muestra:', len(muestra_rutas))

Total de correos disponibles en Enron: 517401
Tamaño de la muestra: 12080


## Extraer texto de la muestra de Enron

In [18]:
import email
from email import policy
import pandas as pd

filas = []
errores = 0

for ruta_archivo in muestra_rutas:
    try:
        with open(ruta_archivo, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)

        subject = msg.get('Subject', '')

        cuerpo = ''
        if msg.is_multipart():
            for parte in msg.walk():
                if parte.get_content_type() == 'text/plain':
                    try:
                        cuerpo += parte.get_content()
                    except Exception:
                        pass
        else:
            try:
                cuerpo = msg.get_content()
            except Exception:
                cuerpo = ''

        texto = f"{subject} {cuerpo}".strip()
        filas.append({'text': texto, 'label': 0})
    except Exception as e:
        errores += 1

df_enron = pd.DataFrame(filas)
print(f'Correos procesados correctamente: {len(df_enron)}')
print(f'Errores: {errores}')
print()
print(df_enron.head(3))

Correos procesados correctamente: 12080
Errores: 0

                                                text  label
0  Re: one last reminder and then I will be quiet...      0
1  FW: LCRA modification Katina the following dea...      0
2  TVA Expected PDP Notice here is a draft.  let ...      0


## Unión de las 3 fuentes en un solo DataFrame

Las 3 fuentes ya tienen el mismo esquema (`text`, `label`), así que se
pueden concatenar directamente sin necesidad de renombrar columnas.

In [19]:
import pandas as pd

df_unido = pd.concat([df_pot, df_nazario, df_enron], ignore_index=True)

print('Total de filas unidas:', len(df_unido))
print()
print('Distribución de etiquetas:')
print(df_unido['label'].value_counts())
print()
print(df_unido.head(3))

Total de filas unidas: 24160

Distribución de etiquetas:
label
1    12080
0    12080
Name: count, dtype: int64

                                                text  label
0  Hiba from Ledger [Ledger](http://links.iterabl...      1
1  [Action Needed] Revise Payment Information - J...      1
2  Herzlichen Glückwunsch! Ihr Edeka-Sofortgewinn...      1


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
ruta_salida = '/content/drive/MyDrive/dataset_unido_raw.csv'
df_unido.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print('Guardado en:', ruta_salida, '| filas totales:', len(df_unido))

Guardado en: /content/drive/MyDrive/dataset_unido_raw.csv | filas totales: 24160
